# Wildfire EO Indicators Extraction

This notebook scans a folder structured like:

```text
data/raw/wildfires/
├── Event_1/
│   ├── before_wildfire/
│   │   ├── ...B04...(Raw).tiff
│   │   ├── ...B08...(Raw).tiff
│   │   └── ...B12...(Raw).tiff
│   └── after_wildfire/
│       ├── ...B04...(Raw).tiff
│       ├── ...B08...(Raw).tiff
│       └── ...B12...(Raw).tiff
└── Event_2/
    └── ...

It extracts a small set of wildfire EO indicators for each event:

- **NDVI** before and after
- **NBR** before and after
- **dNBR**
- **vegetation loss %**
- **burned area %** (based on a dNBR threshold)



In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 6)
pd.set_option("display.max_columns", None)

# Change this path if needed
ROOT_DIR = Path("data/raw/wildfires")

# Folder names expected inside each event folder
BEFORE_FOLDER = "before_wildfire"
AFTER_FOLDER = "after_wildfire"

# Thresholds
VEGETATION_THRESHOLD = 0.2   # NDVI > 0.2 treated as vegetation
BURN_DNBR_THRESHOLD = 0.27   # common moderate-burn threshold

ROOT_DIR


PosixPath('data/raw/wildfires')

In [2]:
def find_band_file(folder: Path, band: str):
    candidates = sorted(folder.glob("*.tif")) + sorted(folder.glob("*.tiff"))
    for f in candidates:
        if re.search(fr"[_-]{band}([_-]|\b)", f.name, flags=re.IGNORECASE) or band.lower() in f.name.lower():
            return f
    raise FileNotFoundError(f"Band {band} not found in {folder}")

def read_band(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    arr = np.where(np.isfinite(arr), arr, np.nan)
    return arr, profile, transform, crs

def resample_to_match(src_arr, src_transform, src_crs, dst_shape, dst_transform, dst_crs):
    dst = np.empty(dst_shape, dtype="float32")
    reproject(
        source=src_arr,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        src_nodata=np.nan,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )
    return dst

def safe_index(num, den):
    out = np.divide(num, den, out=np.full_like(num, np.nan, dtype="float32"), where=np.abs(den) > 1e-6)
    return out

def summarize(arr):
    valid = np.isfinite(arr)
    if valid.sum() == 0:
        return {"mean": np.nan, "median": np.nan, "min": np.nan, "max": np.nan}
    return {
        "mean": float(np.nanmean(arr)),
        "median": float(np.nanmedian(arr)),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
    }


In [3]:
def compute_wildfire_metrics(event_dir: Path):
    before_dir = event_dir / BEFORE_FOLDER
    after_dir = event_dir / AFTER_FOLDER

    # Before
    b4_before_path = find_band_file(before_dir, "B04")
    b8_before_path = find_band_file(before_dir, "B08")
    b12_before_path = find_band_file(before_dir, "B12")

    b4_before, profile, transform, crs = read_band(b4_before_path)
    b8_before, _, _, _ = read_band(b8_before_path)
    b12_before_raw, _, b12_before_transform, b12_before_crs = read_band(b12_before_path)

    # After
    b4_after_path = find_band_file(after_dir, "B04")
    b8_after_path = find_band_file(after_dir, "B08")
    b12_after_path = find_band_file(after_dir, "B12")

    b4_after, _, _, _ = read_band(b4_after_path)
    b8_after, _, _, _ = read_band(b8_after_path)
    b12_after_raw, _, b12_after_transform, b12_after_crs = read_band(b12_after_path)

    # Resample B12 (20 m) to match B8 grid (10 m)
    b12_before = resample_to_match(
        b12_before_raw, b12_before_transform, b12_before_crs,
        b8_before.shape, transform, crs
    )
    b12_after = resample_to_match(
        b12_after_raw, b12_after_transform, b12_after_crs,
        b8_after.shape, transform, crs
    )

    # Indices
    ndvi_before = safe_index(b8_before - b4_before, b8_before + b4_before)
    ndvi_after  = safe_index(b8_after - b4_after, b8_after + b4_after)

    nbr_before = safe_index(b8_before - b12_before, b8_before + b12_before)
    nbr_after  = safe_index(b8_after - b12_after, b8_after + b12_after)

    dnbr = nbr_before - nbr_after

    valid = np.isfinite(ndvi_before) & np.isfinite(ndvi_after) & np.isfinite(nbr_before) & np.isfinite(nbr_after)
    veget_before = valid & (ndvi_before > VEGETATION_THRESHOLD)
    burned = valid & (dnbr >= BURN_DNBR_THRESHOLD)

    veget_pixels_before = int(veget_before.sum())
    burned_pixels = int(burned.sum())
    valid_pixels = int(valid.sum())

    mean_ndvi_before = float(np.nanmean(ndvi_before))
    mean_ndvi_after = float(np.nanmean(ndvi_after))
    mean_nbr_before = float(np.nanmean(nbr_before))
    mean_nbr_after = float(np.nanmean(nbr_after))
    mean_dnbr = float(np.nanmean(dnbr))

    vegetation_loss_pct = np.nan
    if veget_pixels_before > 0:
        remaining_veget = int((veget_before & (ndvi_after > VEGETATION_THRESHOLD)).sum())
        vegetation_loss_pct = 100.0 * (veget_pixels_before - remaining_veget) / veget_pixels_before

    burned_area_pct = np.nan
    if valid_pixels > 0:
        burned_area_pct = 100.0 * burned_pixels / valid_pixels

    metrics = {
        "event": event_dir.name,
        "before_folder": str(before_dir),
        "after_folder": str(after_dir),
        "mean_ndvi_before": mean_ndvi_before,
        "mean_ndvi_after": mean_ndvi_after,
        "ndvi_drop": mean_ndvi_before - mean_ndvi_after,
        "mean_nbr_before": mean_nbr_before,
        "mean_nbr_after": mean_nbr_after,
        "mean_dnbr": mean_dnbr,
        "vegetation_loss_pct": vegetation_loss_pct,
        "burned_area_pct": burned_area_pct,
        "valid_pixels": valid_pixels,
        "burned_pixels": burned_pixels,
        "vegetated_pixels_before": veget_pixels_before,
    }

    arrays = {
        "ndvi_before": ndvi_before,
        "ndvi_after": ndvi_after,
        "nbr_before": nbr_before,
        "nbr_after": nbr_after,
        "dnbr": dnbr,
        "valid": valid,
        "burned": burned,
        "profile": profile,
    }

    return metrics, arrays


In [4]:
event_dirs = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir()])
print("Events found:", [p.name for p in event_dirs])

results = []
arrays_by_event = {}

for event_dir in event_dirs:
    try:
        metrics, arrays = compute_wildfire_metrics(event_dir)
        results.append(metrics)
        arrays_by_event[event_dir.name] = arrays
        print(f"Processed: {event_dir.name}")
    except Exception as e:
        print(f"Skipped {event_dir.name}: {e}")

wildfire_df = pd.DataFrame(results).sort_values("event").reset_index(drop=True)
wildfire_df


Events found: ['Greece', 'SerraDaEstrela', 'Turkey']
Processed: Greece
Processed: SerraDaEstrela
Processed: Turkey


,event,before_folder,after_folder,mean_ndvi_before,mean_ndvi_after,ndvi_drop,mean_nbr_before,mean_nbr_after,mean_dnbr,vegetation_loss_pct,burned_area_pct,valid_pixels,burned_pixels,vegetated_pixels_before
0,Greece,data/raw/wildfires/Greece/before_wildfire,data/raw/wildfires/Greece/after_wildfire,0.451633,0.325689,0.125944,0.295756,0.250493,0.045262,21.062963,13.851496,3264918,452240,2499240
1,SerraDaEstrela,data/raw/wildfires/SerraDaEstrela/before_wildfire,data/raw/wildfires/SerraDaEstrela/after_wildfire,0.475102,0.394355,0.080747,0.196250,0.054354,0.141896,16.362578,17.482265,3565482,623327,3358340
2,Turkey,data/raw/wildfires/Turkey/before_wildfire,data/raw/wildfires/Turkey/after_wildfire,0.355941,0.287107,0.068834,0.176849,0.253227,-0.076336,14.151772,3.680184,3330812,122580,2180356


In [5]:
# Save the summary as CSV
output_csv = Path("data/processed/indicators/wildfire_eo_indicators_summary.csv")
wildfire_df.to_csv(output_csv, index=False)
print(f"Saved summary to: {output_csv.resolve()}")


Saved summary to: /Users/stefanoinfusini/Desktop/SecondYear_MS/SDAI/PROJECT SDAI&NLP/EO2Explain/data/raw/wildfires/wildfire_eo_indicators_summary.csv


In [12]:
# Quick visual check for one event
event_to_plot = wildfire_df.loc[0, "event"] if len(wildfire_df) else None
event_to_plot


'Greece'

In [13]:
def plot_wildfire_report_figure(arrs, event_name, burn_threshold=None):
    """
    Parameters
    ----------
    arrs : dict
        Dictionary containing:
        - ndvi_before
        - ndvi_after
        - nbr_before
        - nbr_after
        - dnbr
        - burned
    event_name : str
        Event label to use in titles.
    burn_threshold : float, optional
        Threshold used to derive the burn mask; kept for documentation/caption use.
    """

    ndvi_before = arrs["ndvi_before"]
    ndvi_after = arrs["ndvi_after"]
    nbr_before = arrs["nbr_before"]
    nbr_after = arrs["nbr_after"]
    dnbr = arrs["dnbr"]
    burned = arrs["burned"]

    ndvi_change = np.where(
        np.isfinite(ndvi_before) & np.isfinite(ndvi_after),
        ndvi_after - ndvi_before,
        np.nan
    )

    # Robust ranges
    finite_dnbr = dnbr[np.isfinite(dnbr)]
    if finite_dnbr.size > 0:
        vmax_dnbr = np.nanpercentile(finite_dnbr, 98)
        vmax_dnbr = max(vmax_dnbr, 0.1)
    else:
        vmax_dnbr = 1.0

    finite_ndvi_change = ndvi_change[np.isfinite(ndvi_change)]
    if finite_ndvi_change.size > 0:
        vmax_ndvi_change = np.nanpercentile(np.abs(finite_ndvi_change), 98)
        vmax_ndvi_change = max(vmax_ndvi_change, 0.05)
    else:
        vmax_ndvi_change = 0.2

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    axes = axes.ravel()

    # 1. NDVI before
    im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[0].set_title(f"{event_name} - NDVI before")
    axes[0].axis("off")
    cbar0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    cbar0.set_label("NDVI")

    # 2. NDVI after
    im1 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-1, vmax=1)
    axes[1].set_title(f"{event_name} - NDVI after")
    axes[1].axis("off")
    cbar1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    cbar1.set_label("NDVI")

    # 3. dNBR
    im2 = axes[2].imshow(dnbr, cmap="hot", vmin=0, vmax=vmax_dnbr)
    axes[2].set_title(f"{event_name} - dNBR")
    axes[2].axis("off")
    cbar2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    cbar2.set_label("dNBR")

    # 4. NBR before
    im3 = axes[3].imshow(nbr_before, cmap="viridis", vmin=-1, vmax=1)
    axes[3].set_title(f"{event_name} - NBR before")
    axes[3].axis("off")
    cbar3 = fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)
    cbar3.set_label("NBR")

    # 5. NBR after
    im4 = axes[4].imshow(nbr_after, cmap="viridis", vmin=-1, vmax=1)
    axes[4].set_title(f"{event_name} - NBR after")
    axes[4].axis("off")
    cbar4 = fig.colorbar(im4, ax=axes[4], fraction=0.046, pad=0.04)
    cbar4.set_label("NBR")

    # 6. Burn mask
    im5 = axes[5].imshow(burned, cmap="gray", vmin=0, vmax=1)
    axes[5].set_title(f"{event_name} - Burn mask")
    axes[5].axis("off")

    fig.suptitle(
        f"{event_name}: wildfire detection from Sentinel-2 vegetation and burn indices",
        fontsize=14,
        y=1.02
    )

    return fig

In [14]:
fig = plot_wildfire_report_figure(arrays_by_event[event_to_plot], event_to_plot)
fig.savefig(f"{event_to_plot}_wildfire_report_figure.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [24]:
import rasterio

path = r"data/raw/floods/Emilia/after_flooding/2023-05-23-00_00_2023-05-23-23_59_Sentinel-2_L2A_B03_(Raw).tiff"

with rasterio.open(path) as src:
    print("Bounds:", src.bounds)
    print("CRS:", src.crs)
    print("Width, Height:", src.width, src.height)
    print("Transform:", src.transform)

Bounds: BoundingBox(left=11.820598, bottom=44.502971, right=12.032391, top=44.573716)
CRS: EPSG:4326
Width, Height: 2358 1105
Transform: | 0.00, 0.00, 11.82|
| 0.00,-0.00, 44.57|
| 0.00, 0.00, 1.00|
